# Performance de ações da B3 · 2021 a 2023
### Análise exploratória de retorno, risco e tendência

**Objetivo.** Comparar o desempenho de 12 ações líquidas da B3 ao longo de três anos completos de pregões,
avaliando retorno, risco, comportamento dos retornos diários, quedas a partir do topo e tendência de preço.

**Dados.** Histórico diário de cotações obtido pela API pública yfinance (01/01/2021 a 31/12/2023),
tratado em um pipeline de três camadas (Bronze, Prata e Ouro). Todos os indicadores usam o fechamento ajustado
por proventos e desdobramentos.

**Perguntas de negócio**
1. Qual seria a rentabilidade normalizada (Base 100) de um investimento em cada ativo no início de 2021?
2. Quais ativos tiveram a melhor relação entre retorno e risco?
3. Como se comportam estatisticamente os retornos diários?
4. Qual foi a maior queda de cada ativo a partir do topo e qual foi o mais resiliente?
5. Como os preços se comportaram em relação às médias móveis de 21 e 50 dias?

In [33]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

from src import config
from src.indicadores import juntar_fato_dim, kpis_carteira, preparar_recorte, resumo_por_ativo

CORES = config.CORES_ATIVOS


def em_pct(tabela, colunas, casas=1):
    """Mostra as colunas em porcentagem, sem alterar os dados."""
    return tabela.style.format({c: f"{{:.{casas}%}}" for c in colunas}).hide(axis="index")

## 1. Qualidade da base

Antes da análise, verifica-se se os dados chegaram completos em cada etapa do pipeline:
**Bronze** (dado bruto, exatamente como veio da fonte), **Prata** (dado limpo e padronizado)
e **Ouro** (dado modelado, com os indicadores calculados). Também são checados valores nulos,
registros duplicados e ocorrências atípicas.

In [34]:
bronze = pd.read_csv(config.ARQ_BRONZE)
prata = pd.read_csv(config.ARQ_PRATA, parse_dates=["dt_pregao"])
fato = pd.read_csv(config.ARQ_OURO_FATO, parse_dates=["dt_pregao"])
dim = pd.read_csv(config.ARQ_OURO_DIM)

pd.DataFrame({
    "camada": ["Bronze", "Prata", "Ouro · fato", "Ouro · dimensão"],
    "linhas": [len(bronze), len(prata), len(fato), len(dim)],
    "colunas": [bronze.shape[1], prata.shape[1], fato.shape[1], dim.shape[1]],
})

,camada,linhas,colunas
0,Bronze,8940,12
1,Prata,8940,17
2,Ouro · fato,8940,14
3,Ouro · dimensão,12,4


In [35]:
print("Nulos nas colunas de cálculo:", int(prata[["vl_fechamento_ajustado", "qt_volume_acoes"]].isna().sum().sum()))
print("Duplicados (ativo + data):", int(prata.duplicated(subset=["cd_ticker", "dt_pregao"]).sum()))
print("Pregões sinalizados com volume zero:", int(prata["fg_volume_zero"].sum()))
print("Pregões sinalizados com variação acima de 15%:", int(prata["fg_variacao_extrema"].sum()))

Nulos nas colunas de cálculo: 0
Duplicados (ativo + data): 0
Pregões sinalizados com volume zero: 3
Pregões sinalizados com variação acima de 15%: 6


A base está íntegra nas três camadas: 8.940 registros (12 ativos × 745 pregões), sem valores nulos nem duplicados.
Três pregões apresentam volume zero com preços repetidos do dia anterior, padrão típico de falha de registro na fonte,
e seis apresentam variação diária superior a 15%. Ambos os casos foram preservados e sinalizados,
sem alteração do dado original.

## 2. Indicadores-chave

Quatro números resumem a carteira formada pelas 12 ações, considerando o mesmo valor aplicado em cada uma no primeiro pregão de 2021:

- **Retorno médio da carteira:** ganho ou perda médio dos ativos no período.
- **Ativo destaque:** a ação com o maior retorno acumulado.
- **Ativo de menor performance:** a ação com o menor retorno acumulado.
- **Volatilidade anualizada média:** o quanto os preços oscilaram, em média, numa base anual. Quanto maior, maior o risco.

In [36]:
base = juntar_fato_dim(fato, dim)
recorte = preparar_recorte(base)
resumo = resumo_por_ativo(base)
kpi = kpis_carteira(resumo)

pd.DataFrame({
    "KPI": ["Retorno médio da carteira", "Ativo destaque", "Ativo de menor performance", "Volatilidade anualizada média"],
    "Valor": [
        f"{kpi['retorno_medio_carteira']:.1%}",
        f"{kpi['ativo_destaque']} ({kpi['retorno_ativo_destaque']:.1%})",
        f"{kpi['ativo_menor_performance']} ({kpi['retorno_ativo_menor_performance']:.1%})",
        f"{kpi['volatilidade_media']:.1%} a.a.",
    ],
})

,KPI,Valor
0,Retorno médio da carteira,58.4%
1,Ativo destaque,PETR4 (228.7%)
2,Ativo de menor performance,BBDC4 (-6.9%)
3,Volatilidade anualizada média,33.6% a.a.


## 3. Rentabilidade normalizada (Base 100)

Ações têm preços muito diferentes entre si, o que dificulta a comparação direta.
A **Base 100** resolve isso: o preço de cada ativo no primeiro pregão passa a valer 100, e a série mostra
quanto esse valor se tornou ao longo do tempo. Um ativo que termina em 150 transformou R$ 100 em R$ 150.

In [37]:
p1 = resumo.sort_values("vl_retorno_acumulado", ascending=False)[
    ["cd_ticker", "ds_setor", "vl_base_100_final", "vl_retorno_acumulado", "vl_retorno_anualizado"]
]
em_pct(p1, ["vl_retorno_acumulado", "vl_retorno_anualizado"]).format({"vl_base_100_final": "{:.2f}"})

cd_ticker,ds_setor,vl_base_100_final,vl_retorno_acumulado,vl_retorno_anualizado
PETR4,"Petróleo, Gás e Biocombustíveis",328.69,2.286894,0.496375
PRIO3,"Petróleo, Gás e Biocombustíveis",308.32,2.083153,0.464291
EMBR3,Bens Industriais,267.50,1.675029,0.395534
GGBR4,Materiais Básicos,130.79,0.307949,0.095192
EGIE3,Utilidade Pública,125.35,0.253474,0.079525
ITUB4,Financeiro,124.02,0.240245,0.075652
VALE3,Materiais Básicos,118.11,0.181087,0.057993
WEGE3,Bens Industriais,103.51,0.035052,0.011737
ABEV3,Consumo não Cíclico,102.57,0.025696,0.008631
RENT3,Consumo Cíclico,99.75,-0.002505,-0.000849


In [38]:
fig = px.line(
    recorte, x="dt_pregao", y="vl_base_100", color="cd_ticker", color_discrete_map=CORES,
    category_orders={"cd_ticker": list(p1["cd_ticker"])},
    title="Base 100: quanto valeriam R$ 100 aplicados em cada ação no início de 2021",
    labels={"dt_pregao": "", "vl_base_100": "Valor (R$)", "cd_ticker": "Ativo"},
    template="plotly_white", height=550,
)
fig.add_hline(y=100, line_dash="dash", line_color="#5B6573", annotation_text="nível 100", annotation_position="bottom right")
fig.update_traces(line_width=2, hovertemplate="%{y:.2f}")
fig.update_layout(hovermode="x unified", legend_title_text="Ativo (do maior ao menor retorno)")
fig.show()

**Principais resultados**
- 9 dos 12 ativos encerraram o período acima do valor aplicado.
- PETR4 (R$ 328,69), PRIO3 (R$ 308,32) e EMBR3 (R$ 267,50) concentraram os maiores retornos.
- BBDC4 (R$ 93,07), KLBN11 (R$ 99,20) e RENT3 (R$ 99,75) terminaram abaixo de 100.
- O desempenho foi concentrado: apenas três ativos superaram R$ 200, enquanto a maioria ficou entre R$ 99 e R$ 131.

Os retornos incluem proventos, já que o cálculo utiliza o fechamento ajustado.

## 4. Retorno x risco

Retorno alto nem sempre é o melhor resultado, se veio acompanhado de oscilações muito fortes.
Esta seção compara os dois lados:

- **Volatilidade anualizada:** desvio padrão dos retornos diários multiplicado por √252 (dias úteis no ano). Mede o risco.
- **Índice de Sharpe empírico:** retorno anualizado dividido pela volatilidade anualizada. Mede quanto retorno o ativo entregou
  por unidade de risco. Nesta versão simplificada, a taxa livre de risco não é descontada, então o índice serve para comparar os ativos entre si.

No gráfico, cada bolha é um ativo; quanto mais acima e mais à esquerda, melhor a relação entre retorno e risco.
O tamanho da bolha representa o volume financeiro médio negociado por dia.

In [39]:
p2 = resumo.sort_values("vl_indice_sharpe_empirico", ascending=False)[
    ["cd_ticker", "vl_retorno_acumulado", "vl_volatilidade_anualizada", "vl_indice_sharpe_empirico"]
]
em_pct(p2, ["vl_retorno_acumulado", "vl_volatilidade_anualizada"]).format({"vl_indice_sharpe_empirico": "{:.2f}"})

cd_ticker,vl_retorno_acumulado,vl_volatilidade_anualizada,vl_indice_sharpe_empirico
PETR4,2.286894,0.394223,1.26
PRIO3,2.083153,0.467809,0.99
EMBR3,1.675029,0.468211,0.84
EGIE3,0.253474,0.187621,0.42
GGBR4,0.307949,0.355563,0.27
ITUB4,0.240245,0.288345,0.26
VALE3,0.181087,0.338997,0.17
WEGE3,0.035052,0.328267,0.04
ABEV3,0.025696,0.254649,0.03
RENT3,-0.002505,0.368891,-0.00


In [40]:
dispersao = resumo.assign(
    volatilidade=resumo["vl_volatilidade_anualizada"] * 100,
    retorno=resumo["vl_retorno_acumulado"] * 100,
    volume_mi=resumo["vl_volume_financeiro_medio"] / 1e6,
)
fig = px.scatter(
    dispersao, x="volatilidade", y="retorno", size="volume_mi", text="cd_ticker",
    color="cd_ticker", color_discrete_map=CORES, size_max=28,
    title="Risco x retorno: PETR4 entregou o maior retorno por unidade de risco",
    labels={"volatilidade": "Volatilidade anualizada (% a.a.)", "retorno": "Retorno acumulado (%)",
            "volume_mi": "Volume financeiro médio (R$ mi/dia)"},
    hover_data={"vl_indice_sharpe_empirico": ":.2f"},
    template="plotly_white", height=550,
)
fig.update_traces(textposition="top center", marker_opacity=0.75, marker_line_width=1, marker_line_color="white")
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.update_layout(showlegend=False)
fig.show()

**Principais resultados**
- PETR4 apresentou a melhor relação entre retorno e risco (Sharpe de 1,26), seguida de PRIO3 (0,99) e EMBR3 (0,84).
- PRIO3 e EMBR3 foram os ativos mais voláteis da amostra (46,8% a.a.).
- EGIE3 combinou retorno moderado (+25,3%) com a menor volatilidade (18,8% a.a.), ocupando a 4ª posição no Sharpe.
- RENT3 (0,00), KLBN11 (-0,01) e BBDC4 (-0,08) registraram Sharpe zero ou negativo, sem compensação pelo risco assumido no período.

## 5. Distribuição dos retornos diários

O retorno diário é a variação do preço de um pregão para o seguinte. Observar a distribuição desses retornos
mostra não só a oscilação típica de cada ativo, mas também a frequência e a intensidade dos dias extremos,
que representam o risco de mercado mais severo.

- **Dispersão:** desvio padrão dos retornos diários.
- **Assimetria:** valores negativos indicam quedas extremas mais intensas que as altas extremas.
- **Curtose (em excesso):** valores acima de zero indicam caudas mais pesadas que a distribuição normal.

In [41]:
p3 = resumo.sort_values("vl_curtose", ascending=False)[
    ["cd_ticker", "vl_retorno_diario_desvio", "vl_retorno_diario_min", "vl_retorno_diario_max", "vl_assimetria", "vl_curtose"]
]
em_pct(p3, ["vl_retorno_diario_desvio", "vl_retorno_diario_min", "vl_retorno_diario_max"], casas=2).format(
    {"vl_assimetria": "{:.2f}", "vl_curtose": "{:.2f}"}
)

cd_ticker,vl_retorno_diario_desvio,vl_retorno_diario_min,vl_retorno_diario_max,vl_assimetria,vl_curtose
ITUB4,0.018164,-0.179643,0.082688,-1.11,13.49
BBDC4,0.019771,-0.173843,0.069052,-0.99,8.99
PETR4,0.024834,-0.207098,0.110291,-0.87,7.63
EMBR3,0.029495,-0.149272,0.160221,0.59,4.24
ABEV3,0.016041,-0.057010,0.097240,0.53,3.46
WEGE3,0.020679,-0.101114,0.083595,0.39,2.30
PRIO3,0.029469,-0.107625,0.178709,0.30,2.19
EGIE3,0.011819,-0.047512,0.054012,0.03,1.61
VALE3,0.021355,-0.075913,0.103957,0.40,1.51
RENT3,0.023238,-0.093868,0.091678,0.18,1.29


In [42]:
def histograma_retornos(ativo):
    dados = recorte[(recorte["cd_ticker"] == ativo)].dropna(subset=["vl_retorno_periodo"])
    fig = px.histogram(
        dados.assign(retorno=dados["vl_retorno_periodo"] * 100), x="retorno",
        marginal="box", nbins=60, color_discrete_sequence=[CORES[ativo]],
        title=f"{ativo}: distribuição dos retornos diários",
        labels={"retorno": "Retorno diário (%)"},
        template="plotly_white", height=420,
    )
    fig.update_layout(yaxis_title="Quantidade de pregões")
    fig.show()


histograma_retornos("ITUB4")

In [43]:
histograma_retornos("EGIE3")

**Principais resultados**
- Todos os ativos apresentam curtose positiva, com eventos extremos mais frequentes do que o previsto por uma distribuição normal.
- ITUB4 (13,49), BBDC4 (8,99) e PETR4 (7,63) têm as caudas mais pesadas e assimetria negativa, concentrando os choques de queda.
- EGIE3 tem a menor dispersão diária (1,18%) e o menor choque negativo (−4,75%).

**Evento corporativo.** A maior queda diária do ITUB4 (−17,96% em 04/10/2021) ocorreu no primeiro pregão após a data de corte
da cisão entre Itaú e XP, quando os acionistas receberam BDRs da XP. Como o fechamento ajustado não incorpora cisões,
a variação não corresponde a uma perda econômica do acionista.

In [44]:
prata[(prata["cd_ticker"] == "ITUB4") & (prata["dt_pregao"].between("2021-09-29", "2021-10-06"))][
    ["dt_pregao", "vl_fechamento", "vl_fechamento_ajustado"]
]

,dt_pregao,vl_fechamento,vl_fechamento_ajustado
3909,2021-09-29,25.507502,18.541248
3910,2021-09-30,25.533979,18.560497
3911,2021-10-01,26.187113,19.045130
3912,2021-10-04,21.482788,15.623812
3913,2021-10-05,22.003531,16.002531
3914,2021-10-06,22.074139,16.053885


## 6. Drawdown máximo

O **drawdown** mede o quanto o preço está abaixo do maior valor já atingido até aquele momento.
O **drawdown máximo** é a pior queda registrada do topo até o fundo e indica o tamanho da perda que um investidor
teria precisado suportar no pior momento. Quanto menor a queda, mais resiliente o ativo.

In [45]:
p4 = resumo.sort_values("vl_drawdown_maximo", ascending=False)[
    ["cd_ticker", "vl_drawdown_maximo", "dt_topo_antes_drawdown", "dt_vale_drawdown"]
]
em_pct(p4, ["vl_drawdown_maximo"]).format({
    "dt_topo_antes_drawdown": "{:%d/%m/%Y}", "dt_vale_drawdown": "{:%d/%m/%Y}",
})

cd_ticker,vl_drawdown_maximo,dt_topo_antes_drawdown,dt_vale_drawdown
EGIE3,-0.176517,12/02/2021,05/08/2021
ABEV3,-0.314160,07/06/2021,27/10/2023
GGBR4,-0.339306,11/05/2021,21/09/2021
PRIO3,-0.343162,14/10/2021,05/01/2022
ITUB4,-0.358692,16/06/2021,30/12/2021
RENT3,-0.383986,14/01/2021,29/10/2021
PETR4,-0.391042,21/10/2022,14/12/2022
KLBN11,-0.393049,24/02/2021,29/09/2022
VALE3,-0.413430,28/07/2021,18/11/2021
BBDC4,-0.460469,07/06/2021,23/03/2023


In [46]:
fig = px.area(
    recorte.assign(queda=recorte["vl_drawdown"] * 100),
    x="dt_pregao", y="queda", facet_col="cd_ticker", facet_col_wrap=3,
    color="cd_ticker", color_discrete_map=CORES,
    category_orders={"cd_ticker": list(p4["cd_ticker"])},
    title="Drawdown: queda a partir do topo ao longo do tempo (do mais resiliente ao menos resiliente)",
    labels={"dt_pregao": "", "queda": "Queda (%)"},
    template="plotly_white", height=900,
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(dtick="M12", tickformat="%Y")
fig.update_layout(showlegend=False)
fig.show()

**Principais resultados**
- EGIE3 foi o ativo mais resiliente, com drawdown máximo de −17,7% (fevereiro a agosto de 2021).
- EMBR3 registrou a maior queda, de −58,3% entre outubro de 2021 e julho de 2022, apesar do retorno acumulado de +167,5% no período.
- WEGE3 (−49,5%) e BBDC4 (−46,0%) completam as maiores quedas.

Os painéis usam a mesma escala para permitir a comparação direta entre os ativos.

## 7. Preço x médias móveis (SMA 21 e SMA 50)

A **média móvel simples (SMA)** é a média dos últimos preços de fechamento e suaviza as oscilações diárias,
evidenciando a tendência. A SMA 21 representa aproximadamente um mês de pregões (curto prazo) e a SMA 50,
cerca de dois meses e meio (médio prazo). Preço acima da média indica cotação acima do patamar recente.

As médias são calculadas sobre o fechamento ajustado e passam a existir a partir do 21º e do 50º pregão de 2021.

In [47]:
p5 = resumo.sort_values("pc_pregoes_acima_sma50", ascending=False)[
    ["cd_ticker", "pc_pregoes_acima_sma50", "qt_cruzamentos_sma21_sma50"]
]
em_pct(p5, ["pc_pregoes_acima_sma50"])

cd_ticker,pc_pregoes_acima_sma50,qt_cruzamentos_sma21_sma50
PETR4,72.3%,17
PRIO3,65.4%,15
EMBR3,57.8%,14
EGIE3,56.3%,23
ITUB4,54.9%,15
VALE3,54.2%,12
RENT3,53.7%,15
GGBR4,53.7%,20
BBDC4,49.6%,17
ABEV3,48.9%,13


In [48]:
def grafico_medias_moveis(ativo):
    serie = recorte[recorte["cd_ticker"] == ativo].melt(
        id_vars="dt_pregao", value_vars=["vl_fechamento_ajustado", "vl_sma_21", "vl_sma_50"],
        var_name="serie", value_name="preco",
    )
    serie["serie"] = serie["serie"].map({
        "vl_fechamento_ajustado": "Fechamento ajustado", "vl_sma_21": "SMA 21", "vl_sma_50": "SMA 50",
    })
    fig = px.line(
        serie, x="dt_pregao", y="preco", color="serie",
        color_discrete_map={"Fechamento ajustado": "#7E57C2", "SMA 21": "#FFB74D", "SMA 50": "#81C784"},
        title=f"{ativo}: fechamento ajustado x SMA 21 x SMA 50",
        labels={"dt_pregao": "", "preco": "Preço (R$)", "serie": ""},
        template="plotly_white", height=450,
    )
    fig.show()


grafico_medias_moveis("PETR4")

**Principais resultados**
- PETR4 fechou acima da SMA 50 em 72,3% dos pregões, a maior proporção da amostra.
- KLBN11 registrou a menor proporção (40,8%).
- EGIE3 teve o maior número de cruzamentos entre a SMA 21 e a SMA 50 (23), indicando ausência de tendência prolongada.

A análise é descritiva e não constitui sinal de compra, venda ou previsão de preço.

## 8. Limitações

- Os resultados descrevem o período de 2021 a 2023 e não representam previsão nem recomendação de investimento.
- Nenhuma relação de causa e efeito foi testada; correlação não implica causalidade.
- O Sharpe empírico não considera a taxa livre de risco.
- O fechamento ajustado do Yahoo Finance não incorpora cisões, o que afeta o ITUB4 em outubro de 2021.
- Três pregões apresentam indício de falha de registro na fonte (volume zero com preço repetido).
- O volume financeiro é estimado a partir do preço de fechamento.
- A amostra reúne 12 ações de grande porte, com 1 a 3 ativos por setor, e não representa o mercado como um todo.

---
*Autora: Anaysa Pereira Lopes · Projeto desenvolvido no curso de Visualização de Dados e Business Intelligence (SENAI/SC).*